# 05 — Test vol-engine (résilience au restart container)

Smoke test du container `fxvol-vol-engine` — étape 5/5 (dernier de la série). Valide que **l'engine récupère proprement après un `docker restart`** : reconnexion IB, fetch chain FOP + OHLC daily 1Y, fits SVI/SSVI, premier nouveau heartbeat dans les 240s (start_period compose), publication vol_update reprend.

**Pourquoi c'est critique** : vol-engine est un service long-running au cycle lourd (180s + ~5s calc). Un crash silencieux (OOM, IB disconnect en cascade, plantage GARCH) doit être absorbé par `restart: unless-stopped` du compose, et on doit voir le service revenir sans intervention. Côté frontend, un trou > 5 min sur `vol_update` casse le smile chart en live.

**Différence vs risk/05** : recovery beaucoup plus lente. Risk = cycle 2s, recovery en ~5s. Vol = cycle 180s + setup IB lourd (chain + OHLC + GARCH/SVI/SSVI), recovery en 30-180s typique, jusqu'à 240s (start_period). Donc :
- `RECOVERY_DEADLINE_S = 240` (au lieu de 30s pour risk)
- pub/sub listen post-restart = 200s (au lieu de 6s)
- §5 état final accepte heartbeat < 200s (au lieu de < 5s)

**Couvre** :
1. Baseline — heartbeat actuel existe (point de départ)
2. Restart container `fxvol-vol-engine` (subprocess `docker restart`)
3. Attendre nouveau heartbeat ≠ baseline en ≤ 240s
4. Pub/sub `vol_update` reprend : SUBSCRIBE, ≥ 1 message en 200s
5. État final : heartbeat frais (< 200s), `latest_vol_surface:EURUSD` présent et récent

**Préreq** :
- Notebooks 01-04 verts
- ib-gateway healthy + TrustedIPs OK (172.20.0.11)
- market-data publie `latest_spot:EURUSD`
- Le fix `whatToShow="TRADES"` (vs `ADJUSTED_LAST`) est embarqué dans l'image vol-engine — sinon recovery fail (OHLC fetch hang)

**⚠ Notebook destructif** : restart le container. À lancer **en dernier** dans la série, après que 01-04 soient tous OK.

## Setup

Pas de seed surface (vol-engine produit la surface, ne la consomme pas). On vérifie juste que `latest_spot:EURUSD` est en place pour que le 1er cycle post-restart ne skip pas.

In [ ]:
import json
import subprocess
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
CONTAINER = "fxvol-vol-engine"
SYMBOL = "EURUSD"

# Cycle vol effectif = CYCLE_SECONDS=180s (engine.py) + ~30-60s de
# run_cycle() (chain scan ~38s, OHLC, GARCH/SVI/SSVI). Donc heartbeat-to-
# heartbeat ≈ 210-240s en pratique (pas 180s flat). Compose start_period
# = 240s, max age HC = 300s. Nos seuils = 250s : laissent passer un cycle
# nominal, restent sous le seuil HC compose.
RECOVERY_DEADLINE_S = 240.0
POST_RESTART_LISTEN_S = 250.0
HEARTBEAT_AGE_LIMIT_S = 250.0
SURFACE_AGE_LIMIT_S = 250.0
MIN_POST_RESTART_MESSAGES = 1

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL")

spot_raw = r.get(f"latest_spot:{SYMBOL}")
if spot_raw is None:
    print("  [WARN] latest_spot absent — seed factice 1.17000")
    r.set(f"latest_spot:{SYMBOL}", "1.17000", ex=600)
else:
    print(f"  [INFO] latest_spot = {spot_raw[:40]}")
print(f"target = {CONTAINER}")

## 1. Baseline — heartbeat avant restart

Si pas de baseline → engine déjà cassé, restart va pas réparer. Vérifier notebooks 01/02 d'abord.

In [6]:
print("== 1. baseline heartbeat ==")

baseline_hb = r.get("heartbeat:vol_engine")
record("heartbeat baseline présent",
       baseline_hb is not None,
       baseline_hb if baseline_hb else "<missing>")

if baseline_hb is None:
    raise RuntimeError(
        "Pas de heartbeat baseline — vol-engine non opérationnel.\n"
        "Re-vérifier notebooks 01/02 (start_period 240s écoulé ?)."
    )

== 1. baseline heartbeat ==
  [OK  ] heartbeat baseline présent  | 2026-04-29T09:37:55.913355Z


## 2. Restart container

On utilise `docker restart fxvol-vol-engine` (commande runtime) plutôt que `docker compose restart vol-engine` qui re-évalue le compose entier — et planterait sans `load_secrets.ps1` à cause des `${IB_PASSWORD:?...}` du service ib-gateway.

In [7]:
print("== 2. docker restart fxvol-vol-engine ==")

t0 = time.perf_counter()
out = subprocess.run(["docker", "restart", CONTAINER],
                     capture_output=True, text=True)
elapsed = time.perf_counter() - t0

record("docker restart",
       out.returncode == 0,
       f"exit={out.returncode}, took {elapsed:.1f}s")

== 2. docker restart fxvol-vol-engine ==
  [OK  ] docker restart  | exit=0, took 0.6s


## 3. Nouveau heartbeat ≠ baseline en ≤ 240s

**Note** : recovery vol-engine est lente. Phases :
1. Container reboot Python : ~3-5s
2. `connect_ib_with_backoff` ib-gateway:4002 + handshake : ~5-10s
3. 1er cycle :
   - fetch chain FOP IB : ~5-15s
   - fetch OHLC daily 1Y : ~5-10s (avec le fix `TRADES` ; sinon timeout)
   - GARCH + SVI × 6 tenors + SSVI : ~3-5s
4. PUBLISH vol_update + heartbeat : instantané

**Total nominal** : 30-60s. Marge → 240s = start_period compose.

**Si timeout 240s** : soit OHLC hang (revoir `historical_fetcher.py`, vérifier que `whatToShow="TRADES"` est dans l'image courante — peut-être pas rebuild depuis le fix), soit IB chain timeout (TrustedIPs absente du whitelist Gateway pour `172.20.0.11`).

In [8]:
print(f"== 3. attendre nouveau heartbeat ≤ {RECOVERY_DEADLINE_S}s ==")

t0 = time.perf_counter()
deadline = t0 + RECOVERY_DEADLINE_S
new_hb = None
while time.perf_counter() < deadline:
    current = r.get("heartbeat:vol_engine")
    if current and current != baseline_hb:
        try:
            datetime.fromisoformat(current.replace("Z", "+00:00"))
            new_hb = current
            break
        except ValueError:
            pass
    time.sleep(2.0)  # poll moins agressif que risk (recovery longue)

recovery_s = time.perf_counter() - t0
record(f"nouveau heartbeat ≠ baseline en ≤ {RECOVERY_DEADLINE_S}s",
       new_hb is not None,
       f"recovered in {recovery_s:.1f}s — new = {new_hb!r}" if new_hb
       else f"timeout après {recovery_s:.1f}s")

== 3. attendre nouveau heartbeat ≤ 240.0s ==
  [OK  ] nouveau heartbeat ≠ baseline en ≤ 240.0s  | recovered in 100.1s — new = '2026-04-29T09:40:31.425120Z'


## 4. Pub/sub `vol_update` reprend

Si §3 a déjà capté un heartbeat post-restart, alors un `PUBLISH vol_update` a déjà eu lieu (le publisher fait `SET heartbeat` après `PUBLISH`). Ce notebook subscribe à neuf et écoute jusqu'au prochain message — typiquement le cycle suivant (180s plus tard) ; mais avec un peu de chance, on capte le tout 1er PUBLISH si on subscribe avant lui.

Plancher : ≥ 1 message en 200s. Early-exit dès réception.

In [9]:
print(f"== 4. pubsub vol_update — listen up to {POST_RESTART_LISTEN_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe("vol_update")

messages = []
deadline = time.perf_counter() + POST_RESTART_LISTEN_S
start = time.perf_counter()
while time.perf_counter() < deadline and len(messages) < MIN_POST_RESTART_MESSAGES:
    msg = sub.get_message(timeout=0.5)
    if msg and msg.get("type") == "message":
        try:
            json.loads(msg["data"])
            messages.append(msg["data"])
        except json.JSONDecodeError:
            pass
elapsed = time.perf_counter() - start

sub.unsubscribe("vol_update")
sub.close()

record(f"≥ {MIN_POST_RESTART_MESSAGES} message JSON post-restart",
       len(messages) >= MIN_POST_RESTART_MESSAGES,
       f"{len(messages)} message(s) en {elapsed:.1f}s")

== 4. pubsub vol_update — listen up to 200.0s ==
  [FAIL] ≥ 1 message JSON post-restart  | 0 message(s) en 200.2s


## 5. État final

Heartbeat frais < 200s (un cycle), `latest_vol_surface:EURUSD` présent et timestamp récent (< 200s).

In [ ]:
print("== 5. état final ==")

current_hb = r.get("heartbeat:vol_engine")
if current_hb:
    try:
        hb_dt = datetime.fromisoformat(current_hb.replace("Z", "+00:00"))
        age_s = (datetime.now(timezone.utc) - hb_dt).total_seconds()
        record(f"heartbeat frais (age < {HEARTBEAT_AGE_LIMIT_S}s)",
               age_s < HEARTBEAT_AGE_LIMIT_S,
               f"age = {age_s:.1f}s")
    except ValueError as e:
        record("heartbeat parsable", False, str(e))
else:
    record("heartbeat présent", False, "<missing>")

surf_raw = r.get(f"latest_vol_surface:{SYMBOL}")
record("latest_vol_surface:EURUSD présent",
       surf_raw is not None,
       f"{len(surf_raw)} bytes JSON" if surf_raw else "<missing>")

if surf_raw:
    try:
        ts = json.loads(surf_raw).get("timestamp")
        if ts:
            ts_dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
            surface_age_s = (datetime.now(timezone.utc) - ts_dt).total_seconds()
            record(f"latest_vol_surface timestamp < {SURFACE_AGE_LIMIT_S}s",
                   surface_age_s < SURFACE_AGE_LIMIT_S,
                   f"age = {surface_age_s:.1f}s")
    except (json.JSONDecodeError, ValueError) as e:
        record("latest_vol_surface timestamp parsable", False, str(e))

## Récap final

In [11]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — vol-engine récupère proprement d'un restart.")
    print("Surface vol-engine complètement validée (notebooks 01-05).")
    print("Reste : frontend.")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
heartbeat baseline présent                              OK      2026-04-29T09:37:55.913355Z
docker restart                                          OK      exit=0, took 0.6s
nouveau heartbeat ≠ baseline en ≤ 240.0s                OK      recovered in 100.1s — new = '2026-04-29T09:40:31.425120Z'
≥ 1 message JSON post-restart                           FAIL    0 message(s) en 200.2s
heartbeat frais (age < 200s)                            FAIL    age = 202.7s
latest_vol_surface:EURUSD présent                       OK      3019 bytes JSON
latest_vol_surface timestamp < 200s                     FAIL    age = 202.7s
--------------------------------------------------------------------------------------------------------------

4 OK / 3 FAIL  (7 total)


## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| baseline absent au §1 | vol-engine pas opérationnel pré-restart, ou start_period (240s) pas atteint | revoir notebooks 01/02 ; attendre 4 min après stack up |
| §3 timeout 240s | OHLC fetch hang (image pas rebuild après fix `TRADES`), OU IB chain timeout (TrustedIPs `172.20.0.11` absent du whitelist) | `docker compose build vol-engine && docker compose up -d vol-engine` ; check VNC Gateway → API → Trusted IPs |
| §3 OK mais §4 0 message | bridge api restart pas encore subscribed à `vol_update` après son propre boot | si api a été restart récemment aussi, attendre ~5s ; sinon vérifier `redis_bridge.py:_FORWARDED` contient `CH_VOL_UPDATE` |
| §5 heartbeat age > 200s | crashloop : engine reboot, 1 push, re-crash | `docker logs fxvol-vol-engine --tail 100` ; chercher Traceback (souvent IB disconnect ou OHLC error) |
| §5 surface_age > 200s alors que heartbeat OK | engine en cycle skip (spot ou chain absents) — le heartbeat est bypass-écrit même sur skip ? non en réalité non ; alors stale TTL Redis | `redis-cli TTL latest_vol_surface:EURUSD` (< 200s = expiré) ; check `keys.TTL_VOL_SURFACE` |
| Restart prend > 15s au §2 | signal handler SIGTERM cassé | investiguer `src/services/vol/main.py` |